# 属性传递、HLO 编辑与 cost model

对应 R09/R10/R11。前六个代码单元在保留的旧 CPU wheel 上执行，仍带 **VERSION-SKEW**。最后两个单元复查匹配源码 wheel 和原生 C++ parser 的已保存证据，不将归档复查当作本内核加载了新 wheel。没有 TPU 编译或运行。详见 [属性说明](attributes-and-cost.md)、[源码 metadata 基线](source-metadata-baseline.md) 与 [roofline](roofline.md)。

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").is_file())
sys.path.insert(0, str(root / "research/software-stack/tools"))
from verify_attributes import verify
capture = root / "artifacts/jax-stack/attributes-cost-002"
result = verify(capture)
print("artifacts verified:", result["artifact_count"])
for case in result["cases"]:
    print(case["case"], "tagged HLO instructions:", case["tagged_hlo_instructions"], "FLOPs:", case["cost"]["flops"])

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


artifacts verified: 60
plain tagged HLO instructions: 0 FLOPs: 384.0
context tagged HLO instructions: 1 FLOPs: 384.0
value tagged HLO instructions: 1 FLOPs: 384.0
call tagged HLO instructions: 1 FLOPs: 384.0
grad-call-same tagged HLO instructions: 2 FLOPs: 792.0
grad-call-drop tagged HLO instructions: 1 FLOPs: 792.0


## 公开 metadata API 与默认成本可同时使用

这里重新执行同形状 matmul。`research_flops=999999` 是一个自定义标签，默认模型仍按形状计算 384 FLOPs。
一个描述可并行维度的提示也不会自动改变完整计算的工作量；若希望算每块成本，需要消费提示并定义分块语义。

In [2]:
import jax, jax.numpy as jnp, numpy as np
from jax.experimental.xla_metadata import set_xla_metadata
with np.load(capture / "inputs.npz", allow_pickle=False) as inputs:
    a, w = inputs["a"], inputs["w"]
def marked(a, w):
    with set_xla_metadata(research_flops=999999):
        return jnp.matmul(a, w, precision=jax.lax.Precision.HIGHEST)
lowered = jax.jit(marked).lower(a, w)
compiled = lowered.compile()
actual = np.asarray(compiled(a, w))
np.testing.assert_allclose(actual, a.astype(np.float64) @ w.astype(np.float64), rtol=2e-5, atol=2e-5)
assert 'research_flops="999999"' in lowered.compiler_ir("hlo").as_hlo_text()
assert compiled.cost_analysis()["flops"] == 384
print(compiled.cost_analysis())

{'bytes accessed1{}': 192.0, 'utilization0{}': 1.0, 'bytes accessedout{}': 96.0, 'utilization1{}': 1.0, 'flops': 384.0, 'bytes accessed0{}': 128.0, 'bytes accessed': 416.0}


## Native HLO 属性 getter/setter

下面只修改内存中的独立 HloModule，不写回源码或既有 capture。它演示私有 native API；任意图改写仍须验证 shape、effects、alias 等约束。

In [3]:
from jaxlib import _hlo
from jax._src.lib import _jax
from jax._src import xla_bridge
module = _hlo.hlo_module_from_text((capture / "plain/exported-hlo.txt").read_text())
dot = next(i for c in module.computations() for i in c.instructions() if i.opcode == _hlo.HloOpcode.kDot)
assert dot.get_frontend_attribute("notebook_tag") is None
dot.set_frontend_attribute("notebook_tag", "in-memory-only")
assert dot.get_frontend_attribute("notebook_tag") == "in-memory-only"
print(dot.to_string())
print(_jax.hlo_module_cost_analysis(xla_bridge.get_backend("cpu"), module))

%dot_general.1 = f32[4,6]{1,0} dot(%a.1, %w.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, operand_precision={highest,highest}, frontend_attributes={notebook_tag="in-memory-only"}
{'flops': 384.0, 'bytes accessed': 416.0, 'utilization0{}': 1.0, 'utilization1{}': 1.0, 'bytes accessed0{}': 128.0, 'bytes accessed1{}': 192.0, 'bytes accessedout{}': 96.0}


## 已执行的 HLO 改写与未知成本

捕获中的 add→subtract 已分别编译执行；第一单元重新解析 opcode，并复算 NumPy 参考。
CPU reference analyzer 对外层 TPU custom-call 返回 -1：这是未知成本，不能得到目标设备的 roofline。

In [4]:
before = np.load(capture / "hlo-edit/before-output.npy", allow_pickle=False)
after = np.load(capture / "hlo-edit/after-output.npy", allow_pickle=False)
with np.load(capture / "inputs.npz", allow_pickle=False) as inputs:
    np.testing.assert_allclose(before - after, 2 * inputs["b"], rtol=2e-5, atol=2e-5)
print(result["hlo_edit"]["results"])
print(result["opaque_custom_call"]["scope"])
print(result["opaque_custom_call"]["cost"])
assert result["opaque_custom_call"]["roofline_available"] is False

[{'case': 'before', 'max_absolute_error': 3.683228411155426e-08}, {'case': 'after', 'max_absolute_error': 7.337873109136694e-08}]
CPU reference analyzer applied to TPU-targeted IR; no libtpu cost analysis or device execution.
{'flops': -1.0, 'bytes accessed': -1.0, 'optimal_seconds': -1.0, 'utilization0{}': 1.0, 'utilization1{}': 1.0, 'bytes accessed0{}': -1.0, 'bytes accessed1{}': -1.0, 'bytes accessedout{}': -1.0}


## 已有模型键：latency_metadata

这是一项在部分 GPU latency estimator 中有消费逻辑的键。先在新 CPU 进程复查标签传递和成本 API，再阅读固定源码中的消费者；CPU 成本不等于 GPU NodeCost。详见 [模型调用链](latency-model.md)。

In [5]:
import subprocess
from datetime import datetime, timezone
latency_capture = root / "artifacts/jax-stack" / ("latency-notebook-" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f"))
command = [str(root / ".venv/bin/python"), "-B", str(root / "research/software-stack/tools/latency_metadata_probe.py"), "--output", str(latency_capture)]
run = subprocess.run(command, cwd=root, text=True, capture_output=True, timeout=180)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)
print(run.stdout.strip())
from verify_latency_metadata import verify as verify_latency
latency_result = verify_latency(latency_capture)
for case in latency_result["cases"]:
    print(case["case"], repr(case["normalized_label"]), case["cpu_compiled_cost"]["flops"], case["cpu_compiled_cost"]["bytes accessed"])
print("verified artifacts:", latency_result["artifact_count"])

{"capture": "latency-notebook-20260914212556134837", "cases": 8, "qualifiers": ["VERSION-SKEW"]}


plain None 384.0 416.0
integer '30000' 384.0 416.0
string '30000' 384.0 416.0
zero '0' 384.0 416.0
negative '-1' 384.0 416.0
fractional '30000.5' 384.0 416.0
invalid 'slow' 384.0 416.0
int64-overflow '9223372036854775808' 384.0 416.0
verified artifacts: 47


In [6]:
contract = json.loads((root / "research/software-stack/tools/latency-model-contract.json").read_text())
print(contract["evidence_level"], contract["source_revision"])
print("Parser input unit:", contract["parser"]["input_unit"])
for reader in contract["direct_readers"]:
    print(reader["source_entry"], reader["opcode_scope"], reader["lookup_priority"])
assert contract["runtime_evidence"]["gpu_estimator_executed"] is False
assert contract["upstream_parser_test"]["executed"] is False
print("GPU estimator、C++ parser 测试与目标 TPU 均未由此 Notebook 执行")

SOURCE-ONLY 496bd4bd49db9ecbffd85da630b49c860b724604
Parser input unit: nanoseconds
xla.gpu-node-cost custom-call after nop check nop first; then metadata for custom-call; otherwise approximate fallback
xla.sol-node-cost all opcodes metadata before opcode/table/fusion branches
GPU estimator、C++ parser 测试与目标 TPU 均未由此 Notebook 执行


`30000` 的字符串能在 CPU HLO 中保留，不代表实测耗时 30 µs。非法或溢出标签被 CPU 接受，也不是 GPU parser 验证通过。可并行维度驱动的真实分块仍需要业务语义、正确的 cost consumer 和目标运行验证。

## 匹配源码的 CPU 复验

采集与 native reader 均在固定镜像中绑定到源码构建 003：14 组 metadata 加 2 组 HLO 改写执行，117 个产物。以下只重新读取已保存的原始文件与数值；完整 native cost 复查必须在该源码环境执行。

In [7]:
from matmul_probe import fingerprint
from verify_extensions import audit
source_result = json.loads((root / "research/software-stack/tools/source-metadata-results.json").read_text())
assert source_result["source_bound_producer_and_verifier"] and source_result["qualifiers"] == []
for name, record in source_result["cases"].items():
    saved = root / record["verified_result"]["path"]
    assert fingerprint(saved) == {k: record["verified_result"][k] for k in ["sha256", "size_bytes"]}
    verified = json.loads(saved.read_text())
    target = root / record["capture"]
    assert audit(target)["artifact_count"] == record["artifact_count"]
    assert verified["build_binding"]["source_build_verified"]
    assert verified["verification_runtime"]["source_build_verified"]
    with np.load(target / "inputs.npz", allow_pickle=False) as inputs:
        a64, w64 = inputs["a"].astype(np.float64), inputs["w"].astype(np.float64)
        reference = a64 @ w64
        gradient = 2 * reference @ w64.T
        bias = inputs["b"].astype(np.float64) if "b" in inputs.files else None
    for case in verified["cases"]:
        expected = gradient if case["case"].startswith("grad-") else reference
        np.testing.assert_allclose(np.load(target / case["case"] / "output.npy"), expected, rtol=2e-5, atol=2e-5)
    if name == "attributes":
        for phase, expected in [("before", reference + bias), ("after", reference - bias)]:
            np.testing.assert_allclose(np.load(target / "hlo-edit" / f"{phase}-output.npy"), expected, rtol=2e-5, atol=2e-5)
    print(name, "source build:", verified["build_binding"]["build_id"], "artifacts:", verified["artifact_count"])
print("保存的源码采集/reader 身份、14 组 metadata 数值及 2 组 HLO 改写输出已复查。")

attributes source build: kickoff-cpu-source-003 artifacts: 66
latency source build: kickoff-cpu-source-003 artifacts: 51
保存的源码采集/reader 身份、14 组 metadata 数值及 2 组 HLO 改写输出已复查。


## 原生 latency parser 的独立证据

固定源码的原测试及 14 个边界样本已在 C++ 目标执行通过。这里重新核对 XML properties、二进制与源码指纹，并重放测试补丁；不在 Notebook 中重新运行 C++ 目标。早期 SOURCE-ONLY contract 的未执行标记属于原快照。详见 [原生解析器说明](latency-parser-native.md)。

In [8]:
from verify_latency_parser import verify as verify_native_parser
native_parser = verify_native_parser()
assert native_parser["source_restored"] and native_parser["new_boundary_cases"] == 14
for name, item in native_parser["observations"].items():
    print(name, repr(item["input"]), "configured cycles/us:", item["cycles_per_microsecond"],
          "observed:", item["observed_cycles"])
print("已核对的 C++ 测试:", native_parser["boundary_run_tests_passed"])
print("GPU consumer、真实调度和 TPU 执行仍需独立验证。")

ClockScale '30000' configured cycles/us: 2 observed: 60.0
Empty '' configured cycles/us: 1 observed: None
Fractional '30000.5' configured cycles/us: 1 observed: None
IntegerMaximum '9223372036854775807' configured cycles/us: 1 observed: 9223372036854776.0
IntegerMinimum '-9223372036854775808' configured cycles/us: 1 observed: -9223372036854776.0
Invalid 'slow' configured cycles/us: 1 observed: None
Missing None configured cycles/us: 1 observed: None
Negative '-1000' configured cycles/us: 1 observed: -1.0
NegativeOverflow '-9223372036854775809' configured cycles/us: 1 observed: None
Positive '30000' configured cycles/us: 1 observed: 30.0
PositiveOverflow '9223372036854775808' configured cycles/us: 1 observed: None
Scientific '3e4' configured cycles/us: 1 observed: None
SubMicrosecond '1' configured cycles/us: 1 observed: 0.001
Zero '0' configured cycles/us: 1 observed: 0.0
已核对的 C++ 测试: 15
GPU consumer、真实调度和 TPU 执行仍需独立验证。
